# SOMYTHRA Twin — Optical PPG Autonomic Stress Model
### Training on WESAD Wearable Optical PPG Benchmark Dataset

This notebook trains a physiological stress classification model for **SOMYTHRA Twin** using legitimate optical Photoplethysmography (PPG) features extracted from the **Empatica E4 wristband** across 15 human subjects.

**Key Research Highlights:**
- **Zero Synthetic Data / Zero Fabrication:** Evaluates real optical blood volume pulse (BVP/PPG) data.
- **4 Core Optical Metrics:** , , , .
- **Strict Generalization:** Evaluated using 15-fold **Leave-One-Subject-Out (LOSO)** cross-validation to guarantee zero subject leakage.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
import joblib

# Dynamic search for features.csv across /kaggle/input and local fallbacks
data_path = None
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    found = list(kaggle_input.rglob("*.csv"))
    for cand in found:
        if "feature" in cand.name.lower():
            data_path = cand
            break

if not data_path:
    for p in [Path("data/wesad/features.csv"), Path("../data/wesad/features.csv"), Path("features.csv")]:
        if p.exists():
            data_path = p
            break

if not data_path:
    raise FileNotFoundError("Dataset not found. Please attach 'wesad-engineered-wrist-features' dataset.")

print(f"Loaded dataset from: {data_path.resolve()}")
df = pd.read_csv(data_path)
print(f"Shape: {df.shape}, Subjects: {df['subject'].nunique()}")
df.head()


## 1. Feature Space & Class Distribution

In [ ]:
FEATURES = ["mean_hr", "rmssd", "pnn50", "lf_hf"]
X = df[FEATURES]
y = (df["label_name"].astype(str).str.lower() == "stress").astype(int).values
groups = df["subject"]

print("Class Counts:")
print(df["label_name"].value_counts())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, feat in zip(axes, FEATURES):
    for label_name, group in df.groupby("label_name"):
        ax.hist(group[feat], bins=20, alpha=0.6, label=label_name, density=True)
    ax.set_title(feat)
    ax.legend()
plt.tight_layout()
plt.show()

## 2. Leave-One-Subject-Out (LOSO) Cross-Validation
Testing the model on 15 completely unseen subjects.

In [ ]:
logo = LeaveOneGroupOut()
results = []
y_true_all, y_pred_all, y_prob_all = [], [], []

for train_idx, test_idx in logo.split(X, y, groups):
    sub = groups.iloc[test_idx].iloc[0]
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    clf = RandomForestClassifier(n_estimators=100, max_depth=6, class_weight="balanced", random_state=42)
    clf.fit(X_train_s, y_train)
    
    pred = clf.predict(X_test_s)
    prob = clf.predict_proba(X_test_s)[:, 1]
    
    results.append({
        "subject": sub,
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, prob) if len(np.unique(y_test)) > 1 else np.nan
    })
    
    y_true_all.extend(y_test)
    y_pred_all.extend(pred)
    y_prob_all.extend(prob)

res_df = pd.DataFrame(results)
print(f"Mean LOSO Accuracy: {res_df['accuracy'].mean()*100:.2f}%")
print(f"Mean LOSO F1-Score: {res_df['f1'].mean():.4f}")
print(f"Mean LOSO ROC-AUC:  {res_df['roc_auc'].dropna().mean():.4f}")
res_df

## 3. Train Production Model and Export Artifact

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

prod_model = RandomForestClassifier(n_estimators=100, max_depth=6, class_weight="balanced", random_state=42)
prod_model.fit(X_scaled, y)

artifact = {
    "model": prod_model,
    "scaler": scaler,
    "feature_columns": FEATURES,
    "classes": ["baseline", "stress"],
    "loso_mean_accuracy": float(res_df["accuracy"].mean()),
    "loso_mean_f1": float(res_df["f1"].mean()),
    "loso_mean_roc_auc": float(res_df["roc_auc"].dropna().mean()),
    "feature_importances": dict(zip(FEATURES, [float(v) for v in prod_model.feature_importances_]))
}

joblib.dump(artifact, "somythra_optical_stress_model.pkl")
print("Saved model artifact to somythra_optical_stress_model.pkl")